<a href="https://colab.research.google.com/github/sethspacedigitalconsulting-sketch/space-123/blob/main/03_real_llm_monitoring_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Monitor Cost, Latency, and Quality for an LLM App

## Session outcome

Participants will generate real LLM responses, collect telemetry, calculate operational metrics, and identify alert conditions.

In [ ]:
import importlib.util
import json
import re
import statistics
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import Any, Dict, List


def install_if_missing(package: str, import_name: str | None = None) -> None:
    import_name = import_name or package.split(">=")[0].replace("-", "_")
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


install_if_missing("transformers>=4.44.0", "transformers")
install_if_missing("accelerate>=0.33.0", "accelerate")
install_if_missing("torch", "torch")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"


def load_llm(model_id: str = MODEL_ID):
    print(f"Loading open-source model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
    device = 0 if torch.cuda.is_available() else -1
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)
    print("Device:", "GPU" if torch.cuda.is_available() else "CPU")
    return tokenizer, generator


tokenizer, generator = load_llm()


def ask_llm(system: str, user: str, max_new_tokens: int = 180, temperature: float = 0.7) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = f"System: {system}\nUser: {user}\nAssistant:"

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]
    return output.strip()


def extract_json(text: str) -> Dict[str, Any]:
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        raise ValueError("No JSON object found in model output.")
    return json.loads(match.group(0))

Loading open-source model: HuggingFaceTB/SmolLM2-135M-Instruct


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Device: GPU


## 1. Define Production-Like Requests

In [ ]:
requests = [
    "Explain why prompt versioning matters in production.",
    "What should we monitor after deploying an AI assistant?",
    "How can evals prevent bad AI releases?",
    "What is a rollback plan for an LLM app?",
    "How do we reduce cost without reducing quality?",
    "What risks come from giving tools to an AI agent?",
]

## 2. Generate Responses and Capture Telemetry

Telemetry should include model, prompt version, latency, approximate token/character usage, and quality signals.

In [ ]:
SYSTEM = """You are an AI operations instructor. Answer in one practical paragraph.
Mention concrete production concerns when relevant: evals, monitoring, rollback, cost, latency, safety, or quality.
"""

telemetry = []

for i, question in enumerate(requests, start=1):
    started = time.time()
    answer = ask_llm(SYSTEM, question, max_new_tokens=160, temperature=0.75)
    latency_ms = int((time.time() - started) * 1000)
    approx_tokens = max(1, len(answer.split()))
    estimated_cost = approx_tokens * 0.000001

    telemetry.append({
        "request_id": f"req-{i:03d}",
        "model_id": MODEL_ID,
        "prompt_version": "ops_instructor@v1",
        "question": question,
        "answer": answer,
        "latency_ms": latency_ms,
        "approx_output_tokens": approx_tokens,
        "estimated_cost_usd": estimated_cost,
        "format_ok": len(answer.strip()) > 20,
        "mentions_operational_concern": bool(re.search(r"eval|monitor|rollback|cost|latency|safety|quality|version", answer, re.I)),
    })

telemetry[0]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'max_new_tokens', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[tra

{'request_id': 'req-001',
 'model_id': 'HuggingFaceTB/SmolLM2-135M-Instruct',
 'prompt_version': 'ops_instructor@v1',
 'question': 'Explain why prompt versioning matters in production.',
 'answer': "Prompt versioning is a crucial technique in production that enables developers to manage multiple projects simultaneously, ensuring that all aspects of a project are up-to-date and aligned with the project's goals. It helps in several ways:\n\nBy allowing developers to create multiple versions of a project, it simplifies the process of managing and merging changes, reducing the risk of introducing new bugs or issues. This approach also helps in identifying and resolving conflicts between versions, which can be a significant source of delays and bugs.\n\nPrompt versioning also facilitates parallel development, enabling multiple developers to work on different aspects of the project simultaneously. This approach reduces the time spent on debugging and troubleshooting, allowing developers to f

## 3. Calculate Monitoring Metrics

In [ ]:
def percentile(values, p):
    values = sorted(values)
    index = int((len(values) - 1) * p)
    return values[index]


latencies = [row["latency_ms"] for row in telemetry]
costs = [row["estimated_cost_usd"] for row in telemetry]

metrics = {
    "requests": len(telemetry),
    "p50_latency_ms": percentile(latencies, 0.50),
    "p95_latency_ms": percentile(latencies, 0.95),
    "mean_latency_ms": round(statistics.mean(latencies), 1),
    "total_estimated_cost_usd": round(sum(costs), 6),
    "mean_estimated_cost_usd": round(statistics.mean(costs), 6),
    "format_compliance_rate": round(sum(row["format_ok"] for row in telemetry) / len(telemetry), 2),
    "operational_relevance_rate": round(sum(row["mentions_operational_concern"] for row in telemetry) / len(telemetry), 2),
}

metrics

{'requests': 6,
 'p50_latency_ms': 6542,
 'p95_latency_ms': 7117,
 'mean_latency_ms': 6820.2,
 'total_estimated_cost_usd': 0.000721,
 'mean_estimated_cost_usd': 0.00012,
 'format_compliance_rate': 1.0,
 'operational_relevance_rate': 0.83}

## 4. Define Alerts

Alerts should be connected to decisions: investigate, roll back, adjust prompt, or improve evals.

In [ ]:
ALERT_RULES = {
    "p95_latency_ms": {"max": 45000, "action": "Check runtime, model size, and prompt length."},
    "format_compliance_rate": {"min": 0.95, "action": "Tighten prompt and add structured output parser."},
    "operational_relevance_rate": {"min": 0.80, "action": "Improve prompt and eval cases."},
}


alerts = []
for metric, rule in ALERT_RULES.items():
    value = metrics[metric]
    if "max" in rule and value > rule["max"]:
        alerts.append({"metric": metric, "value": value, "action": rule["action"]})
    if "min" in rule and value < rule["min"]:
        alerts.append({"metric": metric, "value": value, "action": rule["action"]})

alerts if alerts else "No alerts triggered"

'No alerts triggered'

## 5. Inspect Low-Quality Responses

In [ ]:
for row in telemetry:
    if not row["mentions_operational_concern"]:
        print(row["request_id"], row["question"])
        print(row["answer"])
        print()

req-006 What risks come from giving tools to an AI agent?
When giving tools to an AI agent, several risks can arise. Firstly, the AI can inadvertently misuse or misuse the tools, leading to unintended consequences or errors. For instance, a tool that is designed to help with mundane tasks but fails to meet a specific requirement can lead to waste or wasted resources.

Another risk is the potential for AI to pose a threat to humans. If an AI agent is programmed with malicious intent, it may attempt to steal or alter sensitive information, causing harm to innocent individuals. This could lead to severe consequences, such as data breaches or cyber attacks, that could have far-reaching and devastating effects on society.

Additionally, the AI can also fail to learn or adapt to the user's needs and preferences. If the AI's programming is flawed or biased,



## Hands-on deliverable

Create a short monitoring report with:

- one latency metric
- one quality metric
- one cost metric
- one recommended action if quality drops